# Bird Aero Vibration POC: Structural Deformation Prediction with MeshGraphNet

This notebook demonstrates a data-driven approach to structural deformation prediction using **MeshGraphNet**, a graph neural network architecture originally developed by DeepMind.

**Use case:** Bird Aero develops pods and payloads mounted on military aircraft. These structures are subject to complex vibration and deformation loads during flight. Traditional FEA simulations (e.g., COMSOL) are accurate but slow. This POC shows that a GNN trained on FEA simulation data can learn to predict structural deformations orders of magnitude faster, enabling real-time monitoring and rapid design iteration.

**Dataset:** DeepMind's open-source *deforming plate* dataset — COMSOL FEA simulations of a plate under varying loads.

**Pipeline:** Download raw data -> Build graph representations -> Train MeshGraphNet -> Predict next-step deformation

## 1. Setup

Clone the repository and install dependencies. This cell is designed for Google Colab.

In [ ]:
# Clone repo and install
!git clone https://github.com/orireshef/birdaero-vibration-poc.git
%cd birdaero-vibration-poc
!pip install . -q

import sys
sys.path.insert(0, "src")

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Using device: {device}")

## 2. Download Data

Download the DeepMind deforming plate dataset from Google Cloud Storage. This includes `meta.json` and TFRecord files for train/valid/test splits.

In [ ]:
%%time
from vibration_poc.dataset.config import DatasetConfig
from vibration_poc.dataset.download import download_dataset

config = DatasetConfig()
download_dataset(config)
print("Download complete.")
print(f"Raw data directory: {config.raw_dir}")
!ls -lh {config.raw_dir}

## 3. Preprocess

Convert raw TFRecords into graph representations:
- **Nodes** = mesh vertices (features: world position + node type)
- **Edges** = mesh connectivity from tetrahedral cells (features: relative position + distance)
- **Target** = next-step displacement (world_pos_{t+1} - world_pos_t)

Normalization stats are computed on the training split and applied to all splits.

In [ ]:
%%time
from vibration_poc.dataset.preprocess import preprocess_dataset

preprocess_dataset(config)
print("Preprocessing complete.")

# Print stats: number of graphs per split
from pathlib import Path
for split in config.splits:
    split_dir = config.processed_dir / split
    n_graphs = len(list(split_dir.glob("*.pt")))
    print(f"  {split}: {n_graphs} graphs")

## 4. Inspect Data

Load a single graph and examine its structure. Each graph represents one time-step transition in the simulation.

In [ ]:
from vibration_poc.dataset.dataloader import GraphDataset

ds = GraphDataset(config.processed_dir, "train")
print(f"Training dataset size: {len(ds)} graphs\n")

sample = ds[0]
print("Graph tensors:")
for key, val in sample.items():
    print(f"  {key:20s} shape={str(list(val.shape)):15s} dtype={val.dtype}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize the mesh as a 3D scatter plot
mesh_pos = sample["mesh_pos"].numpy()
y = sample["y"].numpy()
disp_mag = np.linalg.norm(y, axis=1)

fig = plt.figure(figsize=(12, 5))

# Mesh colored by displacement magnitude
ax = fig.add_subplot(121, projection="3d")
sc = ax.scatter(
    mesh_pos[:, 0], mesh_pos[:, 1], mesh_pos[:, 2],
    c=disp_mag, cmap="hot", s=2, alpha=0.8
)
ax.set_title("Mesh (colored by displacement magnitude)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.colorbar(sc, ax=ax, shrink=0.6, label="|displacement|")

# Node type distribution
ax2 = fig.add_subplot(122)
node_types = sample["x"][:, 3].numpy()  # last column is node_type (normalized)
ax2.hist(node_types, bins=30, edgecolor="black")
ax2.set_title("Node Type Distribution (normalized)")
ax2.set_xlabel("Node type value")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"\nMesh stats:")
print(f"  Nodes: {mesh_pos.shape[0]}")
print(f"  Edges: {sample['edge_index'].shape[1]}")
print(f"  Displacement range: [{disp_mag.min():.6f}, {disp_mag.max():.6f}]")

## 5. Create Model

Instantiate MeshGraphNet with POC configuration: hidden_dim=64, num_layers=8.

In [ ]:
from vibration_poc.model.meshgraphnet import MeshGraphNet

model = MeshGraphNet(
    input_dim_nodes=4,   # world_pos (3) + node_type (1)
    input_dim_edges=4,   # relative_pos (3) + distance (1)
    output_dim=3,        # displacement (dx, dy, dz)
    hidden_dim=64,
    num_layers=8,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 6. Train

Train for 5 epochs using the POC config. On Colab with a GPU this should complete in a few minutes.

The `train()` function handles model creation, optimizer setup, and checkpoint saving internally.

In [ ]:
%%time
from vibration_poc.training.trainer import TrainingConfig, train

train_config = TrainingConfig(
    epochs=5,
    learning_rate=1e-4,
    lr_decay=0.9999991,
    hidden_dim=64,
    num_layers=8,
    batch_size=1,
    device=device,
    checkpoint_dir="checkpoints",
    log_interval=1,  # print every epoch for the demo
)

best_checkpoint = train(train_config, config)
print(f"\nBest checkpoint saved to: {best_checkpoint}")

## 7. Evaluate

Load the best checkpoint and compute MSE on the test split.

In [ ]:
import torch.nn.functional as F

# Load best checkpoint
eval_model = MeshGraphNet(
    input_dim_nodes=4,
    input_dim_edges=4,
    output_dim=3,
    hidden_dim=64,
    num_layers=8,
)
eval_model.load_state_dict(torch.load(best_checkpoint, weights_only=True))
eval_device = torch.device(device)
eval_model = eval_model.to(eval_device)
eval_model.eval()

# Evaluate on test set
test_ds = GraphDataset(config.processed_dir, "test")
print(f"Test set size: {len(test_ds)} graphs")

losses = []
num_eval = min(len(test_ds), 50)  # evaluate on up to 50 samples
with torch.no_grad():
    for i in range(num_eval):
        graph = {k: v.to(eval_device) for k, v in test_ds[i].items()}
        pred = eval_model(graph)
        loss = F.mse_loss(pred, graph["y"])
        losses.append(loss.item())

mean_mse = np.mean(losses)
std_mse = np.std(losses)
print(f"\nTest MSE ({num_eval} samples):")
print(f"  Mean: {mean_mse:.6f}")
print(f"  Std:  {std_mse:.6f}")
print(f"  Min:  {min(losses):.6f}")
print(f"  Max:  {max(losses):.6f}")

## 8. Visualize Predictions

Compare ground truth vs. predicted deformation for a test sample. Points are colored by displacement magnitude.

In [ ]:
# Pick a test sample
test_graph = {k: v.to(eval_device) for k, v in test_ds[0].items()}

with torch.no_grad():
    pred_disp = eval_model(test_graph).cpu().numpy()

gt_disp = test_graph["y"].cpu().numpy()
mesh = test_graph["mesh_pos"].cpu().numpy()

gt_mag = np.linalg.norm(gt_disp, axis=1)
pred_mag = np.linalg.norm(pred_disp, axis=1)

# Shared color scale
vmin = min(gt_mag.min(), pred_mag.min())
vmax = max(gt_mag.max(), pred_mag.max())

fig = plt.figure(figsize=(16, 6))

# Ground truth
ax1 = fig.add_subplot(131, projection="3d")
sc1 = ax1.scatter(
    mesh[:, 0], mesh[:, 1], mesh[:, 2],
    c=gt_mag, cmap="hot", s=2, vmin=vmin, vmax=vmax
)
ax1.set_title("Ground Truth")
ax1.set_xlabel("X")
ax1.set_ylabel("Y")
ax1.set_zlabel("Z")

# Predicted
ax2 = fig.add_subplot(132, projection="3d")
sc2 = ax2.scatter(
    mesh[:, 0], mesh[:, 1], mesh[:, 2],
    c=pred_mag, cmap="hot", s=2, vmin=vmin, vmax=vmax
)
ax2.set_title("Predicted")
ax2.set_xlabel("X")
ax2.set_ylabel("Y")
ax2.set_zlabel("Z")

# Error
error_mag = np.abs(gt_mag - pred_mag)
ax3 = fig.add_subplot(133, projection="3d")
sc3 = ax3.scatter(
    mesh[:, 0], mesh[:, 1], mesh[:, 2],
    c=error_mag, cmap="coolwarm", s=2
)
ax3.set_title("Absolute Error")
ax3.set_xlabel("X")
ax3.set_ylabel("Y")
ax3.set_zlabel("Z")

plt.colorbar(sc1, ax=ax1, shrink=0.5, label="|disp|")
plt.colorbar(sc2, ax=ax2, shrink=0.5, label="|disp|")
plt.colorbar(sc3, ax=ax3, shrink=0.5, label="|error|")

plt.suptitle("Deformation: Ground Truth vs Prediction", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Per-node error stats:")
print(f"  Mean absolute error: {error_mag.mean():.6f}")
print(f"  Max absolute error:  {error_mag.max():.6f}")

## 9. Next Steps

This POC demonstrates single-step deformation prediction. The next phase (Phase 4) will extend this with:

- **Autoregressive rollout** — predict multiple time steps by feeding predictions back as inputs, enabling full trajectory forecasting
- **Frequency analysis** — apply FFT to predicted deformation sequences to extract dominant vibration modes and natural frequencies
- **Vibration-specific visualization** — animate deformation over time, overlay mode shapes, and compare predicted vs. simulated frequency spectra
- **Real-world data integration** — adapt the pipeline for Bird Aero's proprietary FEA simulations of pod/payload structures

The MeshGraphNet architecture generalizes well to different mesh topologies, making it suitable for the variety of pod/payload geometries in Bird Aero's product line.